**Fig. 3, step 2: fine-tune Cellpose cyto3 on the lateral slices at learning rates 0.01 and 0.1, permissive regime, 11 stacks**

Needs a GPU.

In [ ]:
# Working directory: data/fig3/ under the repository root, where the Fig. 3 deposit folder is downloaded
# (decomposed_2D/, bootstrap_*_models/, domain-specialized_*/, benchmark_outputs*/, metrics_outputs/).
# The 11 ground-truth stacks are read from ../ground_truth/zebrafish/{development,held_out}/.
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for benchmarking_scripts/
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig3")
print("working directory:", os.getcwd())


In [3]:
try:
    from cellpose import models, io, train, core
    from natsort import natsorted
    import tifffile as tiff
    print('Modules imported successfully')
except RuntimeError as e:
    print(f"Error importing cellpose: {e}")
    print("This might be due to a numpy version mismatch. Try restarting the runtime and running the cells again.")
except ImportError as e:
    print(f"Error importing a required library: {e}")
    print("Please ensure all necessary libraries are installed.")

Modules imported successfully


In [4]:
import numpy as np
import tifffile
from skimage.segmentation import find_boundaries
from skimage.measure import label, regionprops
from skimage.morphology import opening, closing, disk
from typing import Tuple, List
import tqdm
import json


In [5]:
# Training function (as in the other training notebooks; the from-scratch branch starts from the cyto3 weights)

import os
import time
from datetime import timedelta
import matplotlib.pyplot as plt
import numpy as np
from cellpose import models, train


def train_cellpose_model(
        train_data,
        train_labels,
        channels,
        model_save_path,
        n_epochs,
        learning_rate,
        weight_decay,
        model_name,
        batch_size,
        normalize,
        min_train_masks,
        momentum,
        rescale=True,
        SGD=True,
        compute_flows=True,
        use_GPU=True,
        return_model=False,
        val_data=None,
        val_labels=None,
        pretrained_model_path=None
):
    """Train (or fine-tune) one Cellpose model on in-memory slices and save it under
    `model_save_path/models/model_name`.

    If that folder already exists the model is loaded instead of retrained. `pretrained_model_path`
    selects fine-tuning from an existing checkpoint; otherwise the network starts from scratch
    (`model_type=None`). Training uses `cellpose.train.train_seg` with the given SGD settings.
    The training loss curve is saved as `<model_name>_train_loss.npy`, and the validation curve
    as `<model_name>_val_loss.npy` when `val_data` / `val_labels` are passed (they are, for
    monitoring only; model selection happens at the 3D level).
    """
    import os
    import numpy as np
    from cellpose import models, train

    # Ensure output directory exists
    if not os.path.exists(model_save_path):
        print('Model directory does not exist, creating directory...')
        os.makedirs(model_save_path)

    model_save_dir = os.path.join(model_save_path, 'models', model_name)
    if os.path.exists(model_save_dir):
        print(f'Model already exists. Loading instead of training: {model_name}')
        model = models.CellposeModel(
            gpu=use_GPU,
            pretrained_model=model_save_dir
        )
        return model if return_model else None

    # === Load previous model (for fine-tuning) OR train from scratch
    if pretrained_model_path:
        print(f"Fine-tuning from previous model: {pretrained_model_path}")
        model = models.CellposeModel(gpu=use_GPU, pretrained_model=pretrained_model_path)
    else:
        print("Training model from scratch (Finetuning).")
        model = models.CellposeModel(gpu=use_GPU, model_type='cyto3', nchan=2)

    print(f"Training model: {model_name}")

    # === Train the model
    new_model_path, train_loss, val_loss = train.train_seg(
        net=model.net,
        train_data=train_data,
        train_labels=train_labels,
        channels=channels,
        save_path=model_save_path,
        n_epochs=n_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        batch_size=batch_size,
        normalize=normalize,
        min_train_masks=min_train_masks,
        model_name=model_name,
        momentum=momentum,
        rescale=rescale,
        SGD=SGD,
        compute_flows=compute_flows,
        test_data=val_data if val_data is not None else None,
        test_labels=val_labels if val_labels is not None else None
    )

    # Save training and validation loss curves
    np.save(os.path.join(model_save_path, "models", model_name + '_train_loss.npy'), train_loss)
    if val_data or val_labels:
        np.save(os.path.join(model_save_path, "models", model_name + '_val_loss.npy'), val_loss)

    print(f"Trained Model saved to: {model_save_path}")
    return model if return_model else None


In [8]:
def load_exported_dataset(dataset_root):
    """
    Load an exported dataset snapshot from disk.
    """

    dataset_root = Path(dataset_root)

    def load_split(split):
        img_dir = dataset_root / split / "images"
        mask_dir = dataset_root / split / "masks"

        imgs, masks = [], []

        img_files = sorted(img_dir.glob("*_img.tif"))
        for img_path in tqdm.tqdm(img_files, total=len(img_files), desc='loading'):
            mask_path = mask_dir / img_path.name.replace("_img.tif", "_mask.tif")

            img = tifffile.imread(img_path)
            mask = tifffile.imread(mask_path)

            imgs.append(img)
            masks.append(mask)

        return imgs, masks

    train_imgs, train_masks = load_split("train")
    val_imgs, val_masks = load_split("val")
    test_imgs, test_masks = load_split("test")

    with open(dataset_root / "metadata.json") as f:
        metadata = json.load(f)

    return (
        train_imgs, train_masks,
        val_imgs, val_masks,
        test_imgs, test_masks,
        metadata
    )


In [9]:
from pathlib import Path
import tifffile
import json

# the split exported by 1_dense_splits/prepare_dense_2d_splits.ipynb (chunked splitting, chunk size 10)
(
    train_imgs, train_masks,
    val_imgs, val_masks,
    test_imgs, test_masks,
    metadata
) = load_exported_dataset(
    "decomposed_2D/decomposed2D_benchmark-lenient_splits/lateral_dataset_v1"
)

loading: 100%|██████████| 835/835 [12:45<00:00,  1.09it/s]
loading: 0it [00:00, ?it/s]


In [10]:
# Fine-tune from Cellpose cyto3
model = train_cellpose_model(
    train_data=train_imgs,
    train_labels=train_masks,
    channels=[2,1],  # membrane, nucleus
    model_save_path="domain-specialized_finetuned_CPCyto3",   # output folder
    n_epochs=500,
    learning_rate=1e-2,
    weight_decay=1e-5,
    model_name="domain-specialized_lateral_500epoch_scratch_lenient_CPcyto3_Finetuned",
    batch_size=16,
    normalize=True,
    min_train_masks=0,
    momentum=0.9,
    rescale=True,
    SGD=True,
    compute_flows=True,
    use_GPU=True,
    return_model=True,
    val_data=val_imgs,
    val_labels=val_masks,
    pretrained_model_path=None
)


 Training model from scratch (Finetuning).


100%|██████████| 25.3M/25.3M [00:03<00:00, 8.23MB/s]


Training model: domain-specialized_lateral_500epoch_scratch_lenient_CPcyto3_Finetuned


  0%|          | 0/3355 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
 19%|█▉        | 633/3355 [00:00<00:03, 796.58it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
 50%|█████     | 1681/3355 [00:02<00:02, 707.42it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/nump

 Trained Model saved to: domain-specialized_finetuned_CPCyto3


In [11]:
# Fine-tune from Cellpose cyto3
model = train_cellpose_model(
    train_data=train_imgs,
    train_labels=train_masks,
    channels=[2,1],  # membrane, nucleus
    model_save_path="domain-specialized_finetuned_CPCyto3",   # output folder
    n_epochs=500,
    learning_rate=1e-1,
    weight_decay=1e-5,
    model_name="domain-specialized_lateral_500epoch_scratch_lenient_CPcyto3_Finetuned-HLR",
    batch_size=16,
    normalize=True,
    min_train_masks=0,
    momentum=0.9,
    rescale=True,
    SGD=True,
    compute_flows=True,
    use_GPU=True,
    return_model=True,
    val_data=val_imgs,
    val_labels=val_masks,
    pretrained_model_path=None
)


 Training model from scratch (Finetuning).
Training model: domain-specialized_lateral_500epoch_scratch_lenient_CPcyto3_Finetuned-HLR


  0%|          | 0/3355 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
  5%|▍         | 162/3355 [00:00<00:05, 626.41it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
 51%|█████     | 1695/3355 [00:02<00:02, 704.18it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/nump

 Trained Model saved to: domain-specialized_finetuned_CPCyto3
